In [4]:
#First of all we import our libraries 

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt 
%matplotlib inline
import pandas as pd
import random

In [ ]:
#We load our dataset with Pokémon names in different languages
#Obtained from https://www.pokecommunity.com/threads/international-list-of-names-in-csv.460446/
#It's a relatively small dataset for this project (with 1025 Pokémon names)
#So we can already expect the models to not permform incredibly

df = pd.read_csv("pokemon_names.csv")


#We create two lists for our Pokémon names in two different languages, English and French
english = [p.lower() for p in df['en']]
french = [p.lower() for p in df['fr']]
len(df)

1025

In [ ]:
#We then define all of our functions, so we can reuse them for our English and French Pokémon nanmes 

#We keep the block size the same as in Andrej's code
#I attempted to use 4 block size, but the validation loss was worse 
block_size = 3 # context length: how many characters do we take to predict the next one? 

#First of all we build the vocab, which keeps track of all the letters (and symbols) in each language and creates a dictionary to interchange them for numbmers
#We return stoi and itos for later use 
def build_vocab(language): 
    chars = sorted(list(set(''.join(language))))
    stoi = {s:i+1 for i,s in enumerate(chars)}
    stoi['^'] = 0
    itos = {i:s for s,i in stoi.items()}
    return stoi, itos

#This function returns our dataset (now converted into numbers), which we will later split into training, validation and test
def build_dataset(words, stoi):
  X, Y = [], []
  for w in words:
    #print(w)
    context = [0] * block_size
    #I exchanged Andrej's "." for "^", since I had dots in my Pokémon names
    for ch in w + '^':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

#Here is where we do all of the work 
#The vocab_size is just the number of letters that each Pokémon language has + ^, or len(itos)
#I kept the embedding_dim and hidden_dim as adjustable parameters to try to get a better validation loss 

def train_model(language, vocab_size, stoi, embedding_dim, hidden_dim): 
    #Split the dataset into training, validation and test 
    random.seed(42)
    random.shuffle(language)
    n1 = int(0.8*len(language))
    n2 = int(0.9*len(language))

    Xtr, Ytr = build_dataset(language[:n1], stoi)
    Xdev, Ydev = build_dataset(language[n1:n2], stoi)
    Xte, Yte = build_dataset(language[n2:], stoi) 

    #Parameters 
    g = torch.Generator().manual_seed(2147483647) # for reproducibility
    C = torch.randn((vocab_size, embedding_dim), generator=g) 
    W1 = torch.randn((embedding_dim*block_size, hidden_dim), generator=g) #we reduced the number of the hidden dimensions (from 200 to 100 because the dataset is very small, it improved the validation loss)
    b1 = torch.randn(hidden_dim, generator=g)
    W2 = torch.randn((hidden_dim, vocab_size), generator=g)
    b2 = torch.randn(vocab_size, generator=g)
    parameters = [C, W1, b1, W2, b2]

    sum(p.nelement() for p in parameters) # number of parameters in total

    for p in parameters:
        p.requires_grad = True

    lre = torch.linspace(-3, 0, 1000)
    lrs = 10**lre

    lri = []
    lossi = []
    stepi = []

    #Actual training, I kept the number of epochs the same as Andrej's 
    for i in range(200000):

        # minibatch construct
        ix = torch.randint(0, Xtr.shape[0], (32,))

        # forward pass
        emb = C[Xtr[ix]] 
        h = torch.tanh(emb.view(-1, embedding_dim*block_size) @ W1 + b1) 
        logits = h @ W2 + b2 
        loss = F.cross_entropy(logits, Ytr[ix])

        # backward pass
        for p in parameters:
            p.grad = None
        loss.backward()

        # update
        lr = 0.1 if i < 100000 else 0.01
        for p in parameters:
            p.data += -lr * p.grad

    # training loss
    emb = C[Xtr] 
    h = torch.tanh(emb.view(-1, embedding_dim*block_size) @ W1 + b1)
    logits = h @ W2 + b2 
    loss = F.cross_entropy(logits, Ytr)
    print(f'Training loss: {loss}')

    # validation loss
    emb = C[Xdev] 
    h = torch.tanh(emb.view(-1, embedding_dim*block_size) @ W1 + b1) 
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ydev)
    print(f'Validation loss: {loss}')

    # test loss
    emb = C[Xte] 
    h = torch.tanh(emb.view(-1, embedding_dim*block_size) @ W1 + b1)
    logits = h @ W2 + b2 
    loss = F.cross_entropy(logits, Yte)
    print(f'Test loss: {loss}')

    #We return the parameters (C, W1, b1, W2, b2) to be able to use them when sampling
    return parameters 

#Then we sample the model, we generate new names. First just in the language that the model was trained on: 

def sample(parameters, itos): 
    #we redefine the parameters that we got from the train_model() function: 
    C, W1, b1, W2, b2 = parameters 
    
    g = torch.Generator().manual_seed(2147483647 + 10)

    for _ in range(20):
        out = []
        context = [0] * block_size # initialize with all ^^^
        while True:
            emb = C[torch.tensor([context])] # (1,block_size,d)
            h = torch.tanh(emb.view(1, -1) @ W1 + b1)
            logits = h @ W2 + b2
            probs = F.softmax(logits, dim=1)
            ix = torch.multinomial(probs, num_samples=1, generator=g).item()
            context = context[1:] + [ix]
            out.append(ix)
            if ix == 0:
                break
        print(''.join(itos[i] for i in out))

#Then we attempt to do a cross linguistic sampling. 
# Instead of initializing the context with all ^^^, each time we take a random sample from the language the model WASN'T trained on. 
# Then we take the first three characters, turn them into numbers and uses that as the context
#The model will have to generate the rest of the word, from the beginning in the other language 

def crossling_sample(parameters, language_2, itos_lang1, stoi_lang1): 
    g = torch.Generator().manual_seed(2147483647 + 10)
    C, W1, b1, W2, b2 = parameters 
    
    for _ in range(20):
        random_choice = random.choice(language_2)
        chars = random_choice[:3]
        context = [stoi_lang1.get(c, 0) for c in chars] #I got the .get(c,0) from ChatGPT because I was trying a for loop that didn't work
        #I have to use the stoi from language 1, otherwise the indices of the embeddings don't match
        #But language 2 may have additional characters (in the case of French), so we can only take words that have the existing characters in a language
        out = []
        while True:
            emb = C[torch.tensor([context])] # (1,block_size,d)
            h = torch.tanh(emb.view(1, -1) @ W1 + b1)
            logits = h @ W2 + b2
            probs = F.softmax(logits, dim=1)
            ix = torch.multinomial(probs, num_samples=1, generator=g).item()
            context = context[1:] + [ix]
            out.append(ix)
            if ix == 0:
                break
        #we add the beginning of context to the generation
        print(chars + ''.join(itos_lang1[i] for i in out), f'Orginial name: {random_choice}')

In [ ]:
#We train the English model and check the loss
stoi_en, itos_en = build_vocab(english)
stoi_fr, itos_fr = build_vocab(french)
parameters_en = train_model(english, 36, stoi_en, 10, 100)

#With 200 hidden dims: 
#Training loss: 1.239607810974121
#Validation loss: 4.9351325035095215
#Test loss: 5.246767520904541

torch.Size([7104, 3]) torch.Size([7104])
torch.Size([878, 3]) torch.Size([878])
torch.Size([904, 3]) torch.Size([904])
Training loss: 1.5377129316329956
Validation loss: 3.9619431495666504
Test loss: 3.9863216876983643


[tensor([[ 1.0790e+00, -1.0492e-01,  1.2483e+00, -5.0874e+00,  1.0718e+00,
          -5.6789e-01, -6.4636e+00,  5.5431e-01,  7.9009e-01,  3.1747e+00],
         [ 5.9324e-01,  3.7421e+00, -1.5460e+00, -1.3048e+00,  2.7366e+00,
           6.3094e+00,  4.4342e+00, -5.9212e+00,  8.3432e-01,  1.5399e-02],
         [ 1.3845e+00, -1.8696e+00, -1.9851e+00,  5.9144e-01, -1.7007e-01,
           3.5972e+00,  4.1200e+00, -1.8447e+00, -5.8607e-01,  1.5330e+00],
         [-2.9632e+00,  6.0269e-01,  1.3172e+00,  1.1653e+00, -1.9356e+00,
          -2.4965e+00, -1.8083e+00,  1.5123e+00, -6.4047e+00,  2.5741e+00],
         [-4.4300e-01, -7.9302e-01,  8.2808e-02, -2.7312e-01, -2.4512e+00,
           1.1053e+00,  1.4890e+00,  1.2636e+00,  5.9591e-01, -2.2709e+00],
         [ 2.1373e-01,  1.7631e+00,  9.5566e-01, -2.0631e+00, -4.3318e-01,
          -1.5596e+00, -3.1361e-01,  2.1769e-02, -1.1770e-01, -6.5957e-01],
         [ 1.3813e+00, -2.9475e-01, -1.3021e+00, -9.5048e-01,  2.1458e+00,
          -1.4681e+

In [ ]:
#We generate English Pokémon names with the model trained in English Pokémon names
sample(parameters_en, itos_en)

hontotodakiimiogi^
beabam^
shiff^
embaosolycolitternopa^
yagon^
sunkard^
rearorzazzlole^
fure^
audliprotler^
cetretulaswooksr^
stufflygond^
cythert^
mant^
hadowupt^
magneghem^
leditt^
hack^
obstrefobra^
elnatu^
spikelndo^


In [ ]:
#We generate Pokémon names with the model trained in English Pokémon names with French contexts
crossling_sample(parameters_en, french, itos_en, stoi_en)

polem^ Orginial name: polagriffe
pohr^ Orginial name: pohmotte
arrokingg^ Orginial name: arrozard
night^ Orginial name: nigosier
piemiogearowzee^ Orginial name: pierroteknik
skeweon^ Orginial name: skelénox
rhidaspoino^ Orginial name: rhinocorne
frouff^ Orginial name: froussardine
miary^ Orginial name: miaouss
vosoly^ Orginial name: vostourno
drak^ Orginial name: dratatin
floaobulb^ Orginial name: floravol
spea^ Orginial name: spectreval
arbaldoer^ Orginial name: arboliva
obat^ Orginial name: obalie
cocirdier^ Orginial name: coconfort
menek^ Orginial name: mentali
sévenosa^ Orginial name: séviper
poudliprotler^ Orginial name: poussacha
grime^ Orginial name: gringolem


In [ ]:
#We train the French model and check the loss 

stoi_fr, itos_fr = build_vocab(french)
len(stoi_fr)
parameters_fr = train_model(french, 42, stoi_fr, 10, 100)

#With 200 hidden dimensions: 
#Training loss: 1.26067316532135
#Validation loss: 4.621716022491455
#Test loss: 4.545237064361572

#With 100 hidden dimensions: 
#Training loss: 1.574519395828247
#Validation loss: 3.3172051906585693
#Test loss: 3.389030933380127

torch.Size([7569, 3]) torch.Size([7569])
torch.Size([961, 3]) torch.Size([961])
torch.Size([947, 3]) torch.Size([947])
Training loss: 1.574519395828247
Validation loss: 3.3172051906585693
Test loss: 3.389030933380127


[tensor([[ 5.3246e+00,  5.3875e-01, -1.0517e+00, -1.7269e+00, -2.6242e-01,
           8.9307e-01, -3.0977e+00, -1.2510e+00,  1.6980e+00,  2.1038e+00],
         [-1.1653e+00,  2.2005e+00,  4.9351e-01,  1.2134e+00,  1.4957e+00,
           5.4077e+00,  3.2231e+00, -2.7145e+00,  1.0221e+00, -1.0862e+00],
         [ 7.9715e-01,  1.7248e+00, -2.0961e+00, -1.0767e+00, -5.7570e-02,
           5.3095e+00,  6.4860e+00, -1.1999e-01, -8.8240e-01, -2.3947e-01],
         [-3.9232e-01,  1.0663e+00,  1.3025e-01,  2.5056e+00, -1.1483e+00,
          -1.1362e+00, -3.1024e+00,  1.8273e+00, -1.6303e+00,  1.1488e+00],
         [-1.2055e+00, -2.0832e-02, -3.9958e-01, -3.3040e-01, -1.5956e+00,
           1.0955e+00,  1.7204e-01,  7.1853e-01, -5.8147e-01, -1.6658e+00],
         [ 5.6598e-02,  1.0778e+00,  3.5013e-01, -1.2463e+00, -6.5076e-02,
          -1.3759e+00,  2.5747e-02,  2.3736e-01, -7.8454e-01, -6.0320e-01],
         [ 7.3824e-01, -2.4670e-01, -8.4631e-01, -2.2011e+00,  1.5404e+00,
          -9.2291e-

In [46]:
#We generate French Pokémon names with the model trained in French Pokémon names
sample(parameters_fr, itos_fr)

has^
lane^
chaus^
exagielncélôme^
vouton^
tipinoféros^
vipe^
arcachenatum^
pauseilly^
rex^
kable^
béis^
scorne^
lumivodono^
arabaffe^
farfaduupingouret^
vémrapchirp^
melabébé^
qwilpirade^
feu^


In [45]:
#We generate Pokémon names with the model trained in French Pokémon names with English contexts
crossling_sample(parameters_fr, english, itos_fr, stoi_fr)

amolga^ Orginial name: amoongus
lokntkuy^ Orginial name: lokix
caranu^ Orginial name: carvanha
whilkuy^ Orginial name: whirlipede
okingalelle^ Orginial name: okidogi
marigomindaco^ Orginial name: marowak
gre^ Orginial name: greedent
golifourc^ Orginial name: golem
hunnombais^ Orginial name: huntail
espo^ Orginial name: espurr
avagos^ Orginial name: avalugg
tranin^ Orginial name: tranquill
gousse^ Orginial name: gourgeist
vulgueriflurbi^ Orginial name: vullaby
cleus^ Orginial name: clefairy
uxio^ Orginial name: uxie
sme^ Orginial name: smeargle
per^ Orginial name: persian
hatamiégeiste^ Orginial name: hattrem
yverpotte^ Orginial name: yveltal
